# Bronze Layer Overview

This notebook implements the **Bronze layer** — the first stage of the Medallion Architecture (Bronze → Silver → Gold) used in this Olist e-commerce pipeline. The Bronze layer lands the raw source data in the lakehouse exactly as it arrives, with only minimal transformation.

Why raw data is stored with minimal transformation:

* The Bronze layer acts as the landing zone — a faithful, replayable copy of the source CSV files inside the lakehouse.
* Every cleaning decision, cast and business rule is applied later, in Silver. If a Silver transformation turns out to be wrong, the pipeline can be rebuilt from the unchanged Bronze data.
* Profiling the raw data before persisting it surfaces data quality issues (nulls, duplicates, implausible values) early, so the Silver transformations can be designed with full knowledge of what the source actually contains.

The project ingests five Olist datasets, each loaded from a CSV file in `/Volumes/workspace/default/olist_raw/raw_data/`:

* **orders** (`olist_orders_dataset.csv`) — one row per order, with the order identifier, purchase/delivery timestamps and `order_status`.
* **order_items** (`olist_order_items_dataset.csv`) — one row per item in an order, with `order_id`, `order_item_id`, `product_id`, `seller_id`, `shipping_limit_date`, `price` and `freight_value`.
* **products** (`olist_products_dataset.csv`) — one row per product, with `product_id`, its category and physical attributes.
* **customers** (`olist_customers_dataset.csv`) — one row per customer, with `customer_id`, `customer_unique_id` and location fields.
* **sellers** (`olist_sellers_dataset.csv`) — one row per seller, with `seller_id` and location fields.

The notebook processes the datasets in turn: load the CSV into a Spark DataFrame, profile it, persist it as a Bronze Delta table, and verify the persisted table.

In [0]:
# Dependency cell (added during notebook split).
# Imports used by the profiling cells below — mirrors the wildcard import in the
# original olist_full_backup notebook (cell 14) so this notebook runs standalone.
from pyspark.sql.functions import *

# Loading Raw Data

## Source Data Ingestion

The raw Olist datasets arrive as CSV files stored in the Unity Catalog volume `/Volumes/workspace/default/olist_raw/raw_data/`. Each dataset is loaded into a Spark DataFrame with the same `spark.read` pattern:

```python
spark.read.format('csv') \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .load('<source path>')
```

* `header='true'` — treats the first row of the CSV as the column names.
* `inferSchema='true'` — Spark scans the file and infers column types (numeric, string, timestamps) instead of reading every column as a string. Inferred types are only as good as the values Spark samples, so the profiling steps that follow always include a `printSchema()` or explicit checks before the data is trusted.

The five source files and the DataFrames they are loaded into:

* `olist_orders_dataset.csv` → `order_df`
* `olist_order_items_dataset.csv` → `order_items_df`
* `olist_products_dataset.csv` → `product_raw`
* `olist_customers_dataset.csv` → `customer_raw`
* `olist_sellers_dataset.csv` → `seller_raw`

One deliberate exception: the customers dataset is loaded with `inferSchema='false'`, so every column arrives as a string. The raw, string-typed representation is what gets persisted to Bronze — explicit typing is applied downstream rather than at ingestion.

In [0]:
order_df=spark.read.format('csv').option('inferschema','true').option('header','true').load('/Volumes/workspace/default/olist_raw/raw_data/olist_orders_dataset.csv/')

## Initial Data Exploration and Profiling

Each dataset is profiled before (or alongside) being persisted. Inspecting schemas, record counts, null values, distinct counts and column statistics at this stage means data quality issues are discovered at the source — where they are cheap to account for in the Silver transformations — instead of after they have propagated into the Gold layer.

The checks present for each dataset:

**Orders**

* `order_df.display()` and `order_df.printSchema()` — a first look at the data and its inferred types.
* `select('order_id').distinct().count()` — tests whether `order_id` is unique (one row per order) by comparing the distinct count against the row count.
* Nulls in `order_delivered_customer_date`, overall and broken down by `order_status` — delivery dates are legitimately missing for orders that were never delivered, so the status breakdown separates expected nulls from anomalies.
* Orders flagged `delivered` that still have a null customer delivery date — a genuine data quality issue; the notebook's comment notes that this check surfaced 8 such records.

**Order items**

* Total `order_id` count vs distinct `order_id` count — reveals the one-to-many relationship between orders and their items.
* Distinct `order_item_id`, and the distinct combination of `order_id` + `order_item_id` — identifies the composite key that makes each row unique.
* Null counts for every column (`order_id`, `product_id`, `seller_id`, `shipping_limit_date`, `price`, `freight_value`, `order_item_id`).
* `distinct().count()` vs row count — duplicate-row check.
* Min/max of `price` and `freight_value` — range check on the numeric business columns.

**Products**

* Row count vs distinct count and distinct `product_id` — tests the uniqueness of the product key.
* Null counts across all columns, plus targeted inspections of products missing the category/description attributes and products missing dimensions/weight.
* `printSchema()` and min/max statistics for every numeric attribute (`product_name_lenght`, `product_description_lenght`, `product_photos_qty`, `product_weight_g`, `product_length_cm`, `product_height_cm`, `product_width_cm`).
* Rows with `product_weight_g == 0` — physically implausible values that would distort any weight-based analysis.

**Customers**

* Total rows vs distinct `customer_id` and `customer_unique_id` — in the Olist schema `customer_id` is unique per order while `customer_unique_id` identifies the underlying person, so both counts are informative.
* Null counts for the key columns and for every column, a duplicate-`customer_id` check, and repeat customers grouped by `customer_unique_id`.

**Sellers**

* A single aggregate inspection — total rows, distinct `seller_id`, and null counts for `seller_id`, `seller_zip_code_prefix`, `seller_city` and `seller_state`.

One ordering note: for the customers dataset the Bronze table is written **before** these profiling checks run — the checks serve the same purpose, but they act on `customer_raw` after the table already exists.

In [0]:
order_df.display()

In [0]:
order_df.printSchema()

In [0]:
order_df.select('order_id').distinct().count()
# Total no of rows is same as distinct rows

In [0]:
# Lets check for null values
order_df.filter(order_df.order_delivered_customer_date.isNull()).count()

In [0]:
# no of orders based on its status like delivered,cancled,delayed,etc
order_df.groupBy('order_status').count().display()

In [0]:
# Let us get the no of null we get according to each order status
display(order_df.filter(order_df.order_delivered_customer_date.isNull()).groupBy('order_status').count().orderBy('count',ascending=False))

In [0]:
# we found 8 records that we got status as delivered but in customer feedback we got null
order_df.filter((order_df.order_status=='delivered') & (order_df.order_delivered_customer_date.isNull())).display()

## Bronze Delta Table Creation

Each DataFrame is persisted as a Delta table:

* **Delta format** gives the Bronze tables ACID transactional writes, schema tracking and a transaction log, so the ingested data is stored reliably and can be queried by everything downstream — the Silver transformations, SQL, and the validation checks.
* Writing with `saveAsTable` **registers the tables in the Unity Catalog**, so the Bronze layer is a governed, discoverable part of the lakehouse rather than loose files, and the tables are addressable as `workspace.<schema>.<table>`.
* `mode('overwrite')` makes ingestion idempotent — rerunning the notebook rebuilds the Bronze tables from the raw CSVs, which is exactly the replayability the Bronze layer exists to provide.

The tables created and their source datasets:

| Bronze table | Source dataset |
| --- | --- |
| `workspace.default.bronze_orders` | orders (`order_df`) |
| `workspace.olist.bronze_order_items` | order items (`order_items_df`) |
| `workspace.olist.bronze_products` | products (`product_raw`) |
| `workspace.olist.bronze_customers` | customers (`customer_raw`) |
| `workspace.olist.bronze_sellers` | sellers (`seller_raw`) |

The first write cell also runs `CREATE SCHEMA IF NOT EXISTS olist` to ensure the schema exists before the later tables are written into `workspace.olist`. Note that `bronze_orders` is written to the `default` schema, while the remaining four Bronze tables live in `workspace.olist`. The customers write additionally sets `.option('overwriteSchema', 'true')`, which allows the table's schema to be replaced when the table is overwritten.

In [0]:
# creating schema if neede
spark.sql("CREATE SCHEMA IF NOT EXISTS olist")
# creating  bronze delta table
order_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_orders")
spark.sql("SHOW TABLES IN workspace.default").show()


## Bronze Layer Verification

The created tables are verified rather than assumed:

* `bronze_orders` — a `SELECT count(*)` against `workspace.default.bronze_orders`, plus a `display()` of `spark.table("workspace.default.bronze_orders")`, confirm the table exists, is queryable through the catalog, and holds the ingested orders.
* `bronze_order_items` and `bronze_products` — `DESCRIBE DETAIL` returns table-level metadata (format, location, file count, size, creation and modification times), confirming each table was created as a managed Delta table with the expected location.
* `bronze_sellers` — a `count()` on `spark.table('workspace.olist.bronze_sellers')` closes out the final dataset.

These checks close the ingestion loop for each dataset — raw CSV → Spark DataFrame → Delta table in the catalog — with evidence that the data actually landed.

In [0]:
# verifying bronze layer
spark.sql(
    """select count(*) as total_orders
    from workspace.default.bronze_orders"""
)
bronze_orders = spark.table("workspace.default.bronze_orders")

bronze_orders.display()

# Bronze_order_items

In [0]:
order_items_df=spark.read.format('csv').option('inferschema','true').option('header','true').load('/Volumes/workspace/default/olist_raw/raw_data/olist_order_items_dataset.csv/')

In [0]:
# checking if order_id is unique key by calculating total no of order id in this cell and count of distinct order id in next cell
order_items_df.select('order_id').count()


In [0]:
order_items_df.select("order_id").distinct().count()

In [0]:
# distinct no of order_item_id
order_items_df.select('order_item_id').distinct().count()

In [0]:
# count of nulls in all columns
order_items_df.select(
    sum(when(col("order_id").isNull(), 1).otherwise(0)).alias("order_id_null"),
    sum(when(col("product_id").isNull(), 1).otherwise(0)).alias("product_id_null"),
    sum(when(col("seller_id").isNull(), 1).otherwise(0)).alias("seller_id_null"),
    sum(when(col("shipping_limit_date").isNull(), 1).otherwise(0)).alias("shipping_limit_date_null"),
    sum(when(col("price").isNull(), 1).otherwise(0)).alias("price_null"),
    sum(when(col("freight_value").isNull(), 1).otherwise(0)).alias("freight_value_null"),
    sum(
        when(col("order_item_id").isNull(), 1).otherwise(0)
    ).alias("order_item_id_null")
).display()


In [0]:
# no distinct rows in each column
order_items_df.distinct().count()

In [0]:
# distinct combination of order_id,order_item_id
order_items_df.select('order_id','order_item_id').distinct().count()


In [0]:
# min/max value of prices and freight_value
order_items_df.select(
    min(col('price')).alias('min_price'),
    max(col('price')).alias('max_price'),
    min(col('freight_value')).alias('min_freight_value'),
    max(col('freight_value')).alias('max_freight_value')
).display()



In [0]:
# bronze table
order_items_df.write.mode('overwrite').saveAsTable('workspace.olist.bronze_order_items')

In [0]:
%sql
describe detail workspace.olist.bronze_order_items

# Product Bronze

In [0]:
product_raw=spark.read.format('csv').option('header','true').option('inferSchema','true').load('/Volumes/workspace/default/olist_raw/raw_data/olist_products_dataset.csv')

In [0]:
product_raw.display()

In [0]:
product_raw.count()
product_raw.distinct().count()



In [0]:
product_raw.select('product_id').distinct().count()

In [0]:
# Count nulls in each column of product_raw
null_counts = product_raw.select(
    [sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in product_raw.columns]
)
null_counts.display()

In [0]:
#  inspection
product_raw.select("product_id").where(col('product_category_name').isNull()).where(col('product_name_lenght').isNull()).where(col('product_description_lenght').isNull()).where(col('product_photos_qty').isNull()).groupBy(col('product_id')).count().display()



In [0]:
# inspection
product_raw.select(col("product_id")).where(col('product_height_cm').isNull()).where(col('product_length_cm').isNull()).where(col('product_weight_g').isNull()).where(col('product_width_cm').isNull()).groupBy(col('product_id')).count().display()

In [0]:
product_raw.printSchema()

In [0]:
product_raw.agg(
    min('product_name_lenght').alias('min_name_length'),
    max('product_name_lenght').alias('max_name_length'),

    min('product_description_lenght').alias('min_description_length'),
    max('product_description_lenght').alias('max_description_length'),

    min('product_photos_qty').alias('min_photos'),
    max('product_photos_qty').alias('max_photos'),

    min('product_weight_g').alias('min_weight'),
    max('product_weight_g').alias('max_weight'),

    min('product_length_cm').alias('min_length'),
    max('product_length_cm').alias('max_length'),

    min('product_height_cm').alias('min_height'),
    max('product_height_cm').alias('max_height'),

    min('product_width_cm').alias('min_width'),
    max('product_width_cm').alias('max_width')
).display()

In [0]:
# inspecting the rows with zero weight
product_raw.filter(col('product_weight_g')==0).count()

In [0]:
product_raw.filter(col('product_weight_g')==0).select(col('product_id'),col('product_category_name'),col('product_height_cm'),col('product_length_cm'),col('product_weight_g'),col('product_width_cm')).display()

In [0]:
# product bronze table creation
product_raw.write.format('delta').mode('overwrite').saveAsTable('workspace.olist.bronze_products')

In [0]:
%sql
describe detail workspace.olist.bronze_products

# Bronze customer

In [0]:
customer_raw = spark.read.format('csv') \
    .option('header', 'true') \
    .option('inferSchema', 'false') \
    .load('/Volumes/workspace/default/olist_raw/raw_data/olist_customers_dataset.csv')

In [0]:
customer_raw.write.format('delta') \
    .option('overwriteSchema', 'true') \
    .mode('overwrite') \
    .saveAsTable('workspace.olist.bronze_customers')

In [0]:
# Customer dataset profiling
customer_raw.agg(
    count('*').alias('total_rows'),
    countDistinct('customer_id').alias('distinct_customer_ids'),
    countDistinct('customer_unique_id').alias('distinct_unique_customers'),
    sum(when(col('customer_id').isNull(), 1).otherwise(0)).alias('null_customer_id'),
    sum(when(col('customer_unique_id').isNull(), 1).otherwise(0)).alias('null_unique_id')
).display()

# NULL count in every column
customer_raw.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in customer_raw.columns
]).display()

# Check duplicate customer_id records
customer_raw.groupBy('customer_id').count() \
    .filter(col('count') > 1).display()

# Inspect schema
customer_raw.printSchema()

In [0]:


customer_raw.groupBy('customer_unique_id') \
    .count() \
    .filter(col('count') > 1) \
    .orderBy(col('count').desc()) \
    .display()

# Sellers Raw

In [0]:
seller_raw=spark.read.format('csv').option('header','true').option('inferSchema','true').load('/Volumes/workspace/default/olist_raw/raw_data/olist_sellers_dataset.csv')

In [0]:
# General inspection
seller_raw.agg(
    count('*').alias('total_rows'),
    countDistinct('seller_id').alias('distinct_sellers'),
    sum(when(col('seller_id').isNull(), 1).otherwise(0)).alias('null_seller_id'),
    sum(when(col('seller_zip_code_prefix').isNull(), 1).otherwise(0)).alias('null_zip_code'),
    sum(when(col('seller_city').isNull(), 1).otherwise(0)).alias('null_city'),
    sum(when(col('seller_state').isNull(), 1).otherwise(0)).alias('null_state')
).display()

# Seller Bronze

In [0]:
seller_raw.write.format('delta') \
    .mode('overwrite') \
    .saveAsTable('workspace.olist.bronze_sellers')

In [0]:
spark.table('workspace.olist.bronze_sellers').count()